# Lab | Hypothesis Testing

**Objective**

Welcome to the Hypothesis Testing Lab, where we embark on an enlightening journey through the realm of statistical decision-making! In this laboratory, we delve into various scenarios, applying the powerful tools of hypothesis testing to scrutinize and interpret data.

From testing the mean of a single sample (One Sample T-Test), to investigating differences between independent groups (Two Sample T-Test), and exploring relationships within dependent samples (Paired Sample T-Test), our exploration knows no bounds. Furthermore, we'll venture into the realm of Analysis of Variance (ANOVA), unraveling the complexities of comparing means across multiple groups.

So, grab your statistical tools, prepare your hypotheses, and let's embark on this fascinating journey of exploration and discovery in the world of hypothesis testing!

**Challenge 1**

In this challenge, we will be working with pokemon data. The data can be found here:

- https://raw.githubusercontent.com/data-bootcamp-v4/data/main/pokemon.csv

In [1]:
import pandas as pd

# Load the dataset
url = "https://raw.githubusercontent.com/data-bootcamp-v4/data/main/pokemon.csv"
df = pd.read_csv(url)

# Display first few rows
print(df.head())


            Name Type 1  Type 2  HP  Attack  Defense  Sp. Atk  Sp. Def  Speed  \
0      Bulbasaur  Grass  Poison  45      49       49       65       65     45   
1        Ivysaur  Grass  Poison  60      62       63       80       80     60   
2       Venusaur  Grass  Poison  80      82       83      100      100     80   
3  Mega Venusaur  Grass  Poison  80     100      123      122      120     80   
4     Charmander   Fire     NaN  39      52       43       60       50     65   

   Generation  Legendary  
0           1      False  
1           1      False  
2           1      False  
3           1      False  
4           1      False  


- We posit that Pokemons of type Dragon have, on average, more HP stats than Grass. Choose the propper test and, with 5% significance, comment your findings.

In [3]:
from scipy.stats import ttest_ind

# Filter data
dragon_hp = df[df['Type 1'] == 'Dragon']['HP']
grass_hp = df[df['Type 1'] == 'Grass']['HP']

# Two-sample t-test (independent)
t_stat, p_val = ttest_ind(dragon_hp, grass_hp, equal_var=False)

print(f"T-statistic: {t_stat:.4f}")
print(f"P-value: {p_val:.4f}")

# Conclusion at 5% significance
if p_val < 0.05:
    print("Reject the null hypothesis: Dragons have significantly different HP than Grass Pokemons.")
else:
    print("Fail to reject the null hypothesis: No significant difference in HP between Dragons and Grass Pokemons.")


T-statistic: 3.3350
P-value: 0.0016
Reject the null hypothesis: Dragons have significantly different HP than Grass Pokemons.


- We posit that Legendary Pokemons have different stats (HP, Attack, Defense, Sp.Atk, Sp.Def, Speed) when comparing with Non-Legendary. Choose the propper test and, with 5% significance, comment your findings.


In [5]:
from scipy.stats import ttest_ind

# List of stats to compare
stats = ['HP', 'Attack', 'Defense', 'Sp. Atk', 'Sp. Def', 'Speed']

# Split data
legendary = df[df['Legendary'] == True]
non_legendary = df[df['Legendary'] == False]

# Perform t-tests for each stat
for stat in stats:
    t_stat, p_val = ttest_ind(legendary[stat], non_legendary[stat], equal_var=False)
    print(f"{stat}: T-stat={t_stat:.4f}, P-value={p_val:.4f}")
    if p_val < 0.05:
        print(f"  => Significant difference in {stat}")
    else:
        print(f"  => No significant difference in {stat}")


HP: T-stat=8.9814, P-value=0.0000
  => Significant difference in HP
Attack: T-stat=10.4381, P-value=0.0000
  => Significant difference in Attack
Defense: T-stat=7.6371, P-value=0.0000
  => Significant difference in Defense
Sp. Atk: T-stat=13.4174, P-value=0.0000
  => Significant difference in Sp. Atk
Sp. Def: T-stat=10.0157, P-value=0.0000
  => Significant difference in Sp. Def
Speed: T-stat=11.4750, P-value=0.0000
  => Significant difference in Speed


**Challenge 2**

In this challenge, we will be working with california-housing data. The data can be found here:
- https://raw.githubusercontent.com/data-bootcamp-v4/data/main/california_housing.csv

In [7]:
# Load California housing data
url = "https://raw.githubusercontent.com/data-bootcamp-v4/data/main/california_housing.csv"
df = pd.read_csv(url)

# Display first few rows
print(df.head())

   longitude  latitude  housing_median_age  total_rooms  total_bedrooms  \
0    -114.31     34.19                15.0       5612.0          1283.0   
1    -114.47     34.40                19.0       7650.0          1901.0   
2    -114.56     33.69                17.0        720.0           174.0   
3    -114.57     33.64                14.0       1501.0           337.0   
4    -114.57     33.57                20.0       1454.0           326.0   

   population  households  median_income  median_house_value  
0      1015.0       472.0         1.4936             66900.0  
1      1129.0       463.0         1.8200             80100.0  
2       333.0       117.0         1.6509             85700.0  
3       515.0       226.0         3.1917             73400.0  
4       624.0       262.0         1.9250             65500.0  


**We posit that houses close to either a school or a hospital are more expensive.**

- School coordinates (-118, 34)
- Hospital coordinates (-122, 37)

We consider a house (neighborhood) to be close to a school or hospital if the distance is lower than 0.50.

Hint:
- Write a function to calculate euclidean distance from each house (neighborhood) to the school and to the hospital.
- Divide your dataset into houses close and far from either a hospital or school.
- Choose the propper test and, with 5% significance, comment your findings.
 

In [9]:
import numpy as np
from scipy.stats import ttest_ind

# Coordinates
school_coords = (-118, 34)
hospital_coords = (-122, 37)

# Euclidean distance function
def calc_distance(row, coords):
    return np.sqrt((row['longitude'] - coords[0])**2 + (row['latitude'] - coords[1])**2)

# Calculate distances
df['dist_school'] = df.apply(lambda row: calc_distance(row, school_coords), axis=1)
df['dist_hospital'] = df.apply(lambda row: calc_distance(row, hospital_coords), axis=1)

# Mark houses as "close" if distance < 0.5 to either
df['close_to_school_or_hospital'] = (df['dist_school'] < 0.5) | (df['dist_hospital'] < 0.5)

# Divide into two groups
close_prices = df[df['close_to_school_or_hospital']]['median_house_value']
far_prices = df[~df['close_to_school_or_hospital']]['median_house_value']

# Perform two-sample t-test
t_stat, p_val = ttest_ind(close_prices, far_prices, equal_var=False)

print(f"T-statistic: {t_stat:.4f}")
print(f"P-value: {p_val:.4f}")

# Conclusion at 5% significance
if p_val < 0.05:
    print("Reject the null hypothesis: Houses close to a school or hospital are significantly more expensive.")
else:
    print("Fail to reject the null hypothesis: No significant price difference for houses near school or hospital.")


T-statistic: 37.9923
P-value: 0.0000
Reject the null hypothesis: Houses close to a school or hospital are significantly more expensive.
